In [1]:

import os, random, unicodedata
from hashlib import blake2b
from collections import Counter


In [2]:

ROOT = "D:/Tasks/Project_SLM"
NGRAM, BANDS, ROWS = 5, 8, 4
MIN_CHARS, MAX_CHARS = 200, 100_000
MASK = (1 << 61) - 1
random.seed(0)
PERM = [(random.getrandbits(60) | 1, random.getrandbits(60)) for _ in range(BANDS * ROWS)]


In [3]:
SCRIPTS = {
    "te": lambda c: "\u0c00" <= c <= "\u0c7f",
    "de": lambda c: (c.isascii() and c.isalpha()) or c in "äöüÄÖÜß",
    "en": lambda c: c.isascii(),
}
MIN_RATIO = {"te": 0.70, "de": 0.85, "en": 0.95}
FERTILITY = {"en": 4.46, "de": 4.62, "te": 11.03}

FILES = {
    "en": [f"{ROOT}/raw/en2.txt"],
    "de": [f"{ROOT}/raw/de2.txt"],
}

def signature(doc):
    w = doc.split()
    grams = {" ".join(w[i:i+NGRAM]) for i in range(len(w)-NGRAM+1)} or {doc}
    base = [int.from_bytes(blake2b(g.encode(), digest_size=8).digest(), "big") & MASK
            for g in grams]
    mins = [min((a*h+b) & MASK for h in base) for a, b in PERM]
    return [b"".join(m.to_bytes(8, "big") for m in mins[i*ROWS:(i+1)*ROWS])
            for i in range(BANDS)]

def script_ok(doc, lang):
    test = SCRIPTS[lang]
    letters = [c for c in doc if c.isalpha()]
    if not letters:
        return False
    return sum(test(c) for c in letters) / len(letters) >= MIN_RATIO[lang]


def clean_lang(lang):
    seen, bands, s = set(), [set() for _ in range(BANDS)], Counter()
    os.makedirs(f"{ROOT}/clean", exist_ok=True)
    with open(f"{ROOT}/clean/{lang}.txt", "w", encoding="utf-8") as fout:
        for path in FILES[lang]:
            if not os.path.exists(path):
                print(f"--- {path}  MISSING, skipped"); continue
            print(f"--- {path}", flush=True)
            with open(path, encoding="utf-8") as fin:
                for line in fin:
                    s["in"] += 1
                    doc = unicodedata.normalize("NFC", line.strip())
                    if not (MIN_CHARS <= len(doc) <= MAX_CHARS):
                        s["drop_len"] += 1; continue
                    h = blake2b(doc.encode(), digest_size=16).digest()
                    if h in seen:
                        s["drop_exact"] += 1; continue
                    seen.add(h)
                    if not script_ok(doc, lang):
                        s["drop_script"] += 1; continue
                    sig = signature(doc)
                    if sum(b in bands[i] for i, b in enumerate(sig)) >= 2:
                        s["drop_near"] += 1; continue
                    for i, b in enumerate(sig):
                        bands[i].add(b)
                    fout.write(doc + "\n")
                    s["kept"] += 1; s["bytes"] += len(doc.encode())
                    if s["in"] % 200_000 == 0:
                        print(f"  {s['in']:,} read, {s['kept']:,} kept, "
                              f"{s['bytes']/1e9:.2f} GB", flush=True)

    fert = {"en": 4.46, "de": 4.62, "te": 11.03}[lang]
    print(f"\n{lang}: in={s['in']:,} kept={s['kept']:,} "
          f"({s['kept']/max(s['in'],1):.1%}) {s['bytes']/1e9:.2f} GB")
    for k in ("drop_len", "drop_exact", "drop_script", "drop_near"):
        print(f"   {k:<12}{s[k]:>12,}")
    print(f"=> ~{s['bytes']/fert/1e9:.2f}B tokens (o200k)")
    return s


if __name__ == "__main__":
    for lang in ("en", "de"):
        clean_lang(lang)

--- D:/Tasks/Project_SLM/raw/en2.txt
  200,000 read, 199,324 kept, 0.92 GB
  400,000 read, 398,182 kept, 1.84 GB
  600,000 read, 596,592 kept, 2.77 GB
  800,000 read, 794,190 kept, 3.68 GB
  1,000,000 read, 991,460 kept, 4.59 GB
  1,200,000 read, 1,187,956 kept, 5.50 GB
  1,400,000 read, 1,384,137 kept, 6.41 GB
  1,600,000 read, 1,579,655 kept, 7.31 GB
  1,800,000 read, 1,774,860 kept, 8.21 GB
  2,000,000 read, 1,969,327 kept, 9.12 GB
  2,200,000 read, 2,163,355 kept, 10.02 GB
  2,400,000 read, 2,356,905 kept, 10.91 GB
  2,600,000 read, 2,549,715 kept, 11.80 GB
  2,800,000 read, 2,742,329 kept, 12.68 GB
  3,000,000 read, 2,934,609 kept, 13.58 GB
  3,200,000 read, 3,125,534 kept, 14.44 GB
  3,400,000 read, 3,315,102 kept, 15.29 GB
  3,600,000 read, 3,504,331 kept, 16.13 GB
  3,800,000 read, 3,692,614 kept, 16.97 GB
  4,000,000 read, 3,880,046 kept, 17.82 GB
  4,200,000 read, 4,067,221 kept, 18.66 GB
  4,400,000 read, 4,253,823 kept, 19.52 GB

en: in=4,431,410 kept=4,283,307 (96.7%) 19.6